# FarrahAI — Notebook 5: Full Pipeline Demo

This notebook runs the **complete end-to-end pipeline**:

```
Upload File
    → Preprocess (CLAHE, denoise, threshold)
    → OCR (PaddleOCR)
    → Clean + Chunk
    → Embed (sentence-transformers)
    → Index (FAISS)
    → Student asks question
    → Retrieve top-k chunks
    → Generate answer via Ollama
    → Predict important topics
    → Generate sample paper
```

Run this for your live demo.


In [ ]:
import sys
sys.path.insert(0, '..')

from config.settings import DATA_DIR, EMBEDDINGS_DIR, TEACHER_DB_PATH, OLLAMA_MODEL
from modules.room_manager import create_room, list_rooms, upload_and_index
from modules.teacher_profile import load_teacher_db, add_paper_to_teacher, save_teacher_db, get_teacher_summary
from modules.retriever import retrieve, format_retrieved_context
from modules.ollama_chat import answer_from_notes, is_ollama_running, list_available_models
from modules.predictor import predict_important_topics, generate_sample_paper, format_paper_output

print('FarrahAI Pipeline Ready ✓')
print(f'Ollama running: {is_ollama_running()}')
if is_ollama_running():
    print(f'Available models: {list_available_models()}')

## 1. Create a Subject Room

In [ ]:
# Create teacher profile
db = load_teacher_db(str(TEACHER_DB_PATH))

# Add past paper data for teacher (from Notebook 4)
if 'Prof_Sharma' not in db:
    add_paper_to_teacher(db, 'Prof_Sharma', {
        'subject': 'AI_ML', 'paper_type': 'endsem', 'year': 2024, 'semester': 'odd',
        'total_marks': 100,
        'topics': [
            {'name': 'Neural Networks',  'marks': 15},
            {'name': 'Model Evaluation', 'marks': 10},
            {'name': 'Ensemble Methods', 'marks': 10},
            {'name': 'Clustering',       'marks': 5},
            {'name': 'Deep Learning',    'marks': 10},
        ]
    })
    save_teacher_db(db, str(TEACHER_DB_PATH))

# Create subject room
create_room('AI_ML', 'Prof_Sharma', str(DATA_DIR))

print('\nExisting rooms:')
for r in list_rooms(str(DATA_DIR)):
    print(f"  • {r['subject']} → {r['teacher']}")

## 2. Upload Notes

Change the path to your actual notes file (image or PDF).


In [ ]:
# For demo: create a text file with sample notes if you don't have an image yet
import os
from pathlib import Path

sample_notes_path = '../data/raw/sample_notes.txt'
Path('../data/raw').mkdir(parents=True, exist_ok=True)

with open(sample_notes_path, 'w') as f:
    f.write("""
Unit 1: Introduction to Machine Learning

Machine learning is a subset of artificial intelligence that enables systems to learn from data.
Supervised learning uses labeled training data to build a model.
Unsupervised learning discovers hidden patterns without labeled data.
Reinforcement learning trains agents through reward and punishment signals.

Unit 2: Neural Networks and Deep Learning

A neural network consists of layers of interconnected neurons.
Backpropagation algorithm computes gradients using chain rule of calculus.
Gradient descent optimization minimizes the loss function iteratively.
Activation functions like ReLU, sigmoid, and tanh introduce non-linearity.
Dropout regularization randomly deactivates neurons to prevent overfitting.
Batch normalization normalizes intermediate layer inputs for stable training.

Unit 3: Model Evaluation

Accuracy measures the fraction of correctly classified samples.
Precision is the ratio of true positives to total predicted positives.
Recall measures how many actual positives were correctly retrieved.
F1 score is the harmonic mean of precision and recall.
Cross validation divides data into k folds for robust evaluation.
Confusion matrix shows true vs predicted class distribution.

Unit 4: Ensemble Methods

Random forest builds multiple decision trees using bagging technique.
XGBoost uses gradient boosting with regularization to improve accuracy.
Boosting combines weak learners sequentially to create strong classifier.
Feature importance in XGBoost shows which variables drive predictions.

Unit 5: Clustering and Unsupervised Learning

K-means clustering partitions data into k groups by minimizing inertia.
Silhouette score evaluates how well points fit in their assigned cluster.
DBSCAN detects clusters of arbitrary shape and marks outliers as noise.
Hierarchical clustering builds tree structure showing nested clusters.
""")

# Upload and index
upload_and_index(
    file_path=sample_notes_path,
    subject='AI_ML',
    base_dir=str(DATA_DIR)
)
print('Notes uploaded and indexed ✓')

## 3. Ask a Question

In [ ]:
QUESTION = "What is backpropagation and how does it work?"
# Try: "Explain F1 score"
# Try: "What are the types of clustering algorithms?"
# Try: "How does XGBoost work?"

print(f'Question: {QUESTION}\n')

# Retrieve relevant chunks
results = retrieve(
    query=QUESTION,
    subject='AI_ML',
    index_dir=str(EMBEDDINGS_DIR),
    top_k=5
)

context = format_retrieved_context(results)
print('── Retrieved Note Chunks ──────────────────────────────')
print(context[:800], '...' if len(context) > 800 else '')

In [ ]:
# Generate answer via Ollama (must be running)
if is_ollama_running():
    print(f'Asking Ollama [{OLLAMA_MODEL}]...\n')
    answer = answer_from_notes(QUESTION, context, model=OLLAMA_MODEL)
    print('── ANSWER ──────────────────────────────────────────────')
    print(answer)
else:
    print('Ollama not running. Start with: ollama serve')
    print('Showing raw retrieved context instead:')
    print(context)

## 4. Predict Important Topics

In [ ]:
db = load_teacher_db(str(TEACHER_DB_PATH))

print('── Teacher Summary ──────────────────────────────────────')
print(get_teacher_summary('Prof_Sharma', db))

print('\n── Predicted Important Topics ───────────────────────────')
topics = predict_important_topics(
    teacher_name='Prof_Sharma',
    subject='AI_ML',
    teacher_db=db,
    index_dir=str(EMBEDDINGS_DIR),
    top_n=8
)

for t in topics:
    print(f"  {t['rank']}. {t['topic']}  (appeared {t['historical_count']}x)")
    if t['supporting_notes'] != 'No notes found for this topic.':
        preview = t['supporting_notes'][:120].replace('\n', ' ')
        print(f"     Notes: {preview}...")

## 5. Generate Sample Predicted Paper

In [ ]:
paper = generate_sample_paper(
    teacher_name='Prof_Sharma',
    subject='AI_ML',
    teacher_db=db,
    total_marks=100,
    paper_type='endsem'
)
print(format_paper_output(paper))

## Summary of Metrics

Run Notebooks 1–4 to generate full evaluation metrics.
Below is a quick summary print.


In [ ]:
print('══════════════════════════════════════════════════')
print('  FarrahAI — Evaluation Summary')
print('══════════════════════════════════════════════════')
print('  See outputs/ folder for all charts and CSVs.')
print()
print('  Metric files generated:')
import os
for f in sorted(os.listdir('../outputs')):
    print(f'    • {f}')